# Laboratorio 13 — Datos Geográficos y Análisis Espacial
**Curso:** Minería de Datos (EIN132A25)

## Objetivos
- Trabajar con coordenadas y datos espaciales
- Crear mapas interactivos con **plotly**
- Calcular distancias geográficas (Haversine)
- Aplicar clustering espacial

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

np.random.seed(42)

cidades = {
    "Santiago":    (-33.45, -70.67),
    "Valparaíso":  (-33.04, -71.62),
    "Concepción":  (-36.82, -73.05),
    "La Serena":   (-29.91, -71.25),
    "Antofagasta": (-23.65, -70.40),
    "Temuco":      (-38.73, -72.59),
    "Iquique":     (-20.21, -70.14),
    "Puerto Montt":(-41.47, -72.94),
    "Rancagua":    (-34.17, -70.74),
    "Talca":       (-35.43, -71.67),
}

sucursales = []
for ciudad, (lat, lon) in cidades.items():
    n = np.random.randint(3, 15)
    for i in range(n):
        sucursales.append({
            "ciudad":   ciudad,
            "nombre":   f"Sucursal {ciudad} {i+1}",
            "lat":      lat + np.random.normal(0, 0.05),
            "lon":      lon + np.random.normal(0, 0.05),
            "clientes": np.random.randint(500, 5000),
            "tipo":     np.random.choice(["Premium", "Estándar", "Express"])
        })

df = pd.DataFrame(sucursales)
print(f"Total sucursales: {len(df)}")
df.head()

## 1. Mapa de puntos con plotly

In [ ]:
fig = px.scatter_mapbox(
    df, lat="lat", lon="lon",
    color="tipo", size="clientes",
    hover_name="nombre",
    hover_data={"ciudad": True, "clientes": True, "lat": False, "lon": False},
    color_discrete_map={"Premium": "gold", "Estándar": "steelblue", "Express": "salmon"},
    zoom=4, center={"lat": -33.5, "lon": -70.6},
    title="Sucursales Bancarias en Chile",
    mapbox_style="open-street-map"
)
fig.update_layout(height=600)
fig.show()

## 2. Mapa de calor geográfico

In [ ]:
fig = px.density_mapbox(
    df, lat="lat", lon="lon", z="clientes",
    radius=30, zoom=4, center={"lat": -33.5, "lon": -70.6},
    mapbox_style="open-street-map",
    title="Densidad de clientes por zona",
    color_continuous_scale="YlOrRd"
)
fig.update_layout(height=600)
fig.show()

## 3. Análisis por ciudad

In [ ]:
resumen = df.groupby("ciudad").agg(
    n_sucursales=("nombre", "count"),
    total_clientes=("clientes", "sum"),
    promedio_clientes=("clientes", "mean")
).reset_index().sort_values("total_clientes", ascending=False)
print(resumen)

fig = px.bar(resumen, x="ciudad", y="total_clientes", color="n_sucursales",
             title="Total clientes por ciudad")
fig.show()

## 4. Distancia de Haversine

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def distancia_haversine(lat1, lon1, lat2, lon2):
    """Distancia en km entre dos puntos geográficos."""
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c

lat_stgo, lon_stgo = cidades["Santiago"]
lat_valpo, lon_valpo = cidades["Valparaíso"]
dist = distancia_haversine(lat_stgo, lon_stgo, lat_valpo, lon_valpo)
print(f"Distancia Santiago-Valparaíso: {dist:.1f} km")

## 5. Sucursal más cercana

In [ ]:
def sucursal_mas_cercana(lat_ref, lon_ref, df_sucursales):
    df_s = df_sucursales.copy()
    df_s["distancia_km"] = df_s.apply(
        lambda row: distancia_haversine(lat_ref, lon_ref, row["lat"], row["lon"]), axis=1
    )
    return df_s.nsmallest(3, "distancia_km")[["nombre", "ciudad", "tipo", "distancia_km"]]

resultado = sucursal_mas_cercana(-33.45, -70.67, df)
print("Sucursales más cercanas al centro de Santiago:")
print(resultado)

## 6. Clustering espacial con K-Means

In [ ]:
from sklearn.cluster import KMeans

coords = df[["lat", "lon"]].values
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df["zona"] = kmeans.fit_predict(coords)

fig = px.scatter_mapbox(
    df, lat="lat", lon="lon", color="zona",
    color_continuous_scale="Viridis",
    hover_name="nombre", zoom=4,
    center={"lat": -33.5, "lon": -70.6},
    mapbox_style="open-street-map",
    title="Zonas de cobertura (K-Means espacial, K=5)"
)
fig.update_layout(height=600)
fig.show()

## Ejercicios

### Ejercicio 1 — Matriz de distancias

In [ ]:
ciudad_lista = list(cidades.keys())
matriz = pd.DataFrame(index=ciudad_lista, columns=ciudad_lista, dtype=float)

for c1 in ciudad_lista:
    for c2 in ciudad_lista:
        lat1, lon1 = cidades[c1]
        lat2, lon2 = cidades[c2]
        matriz.loc[c1, c2] = distancia_haversine(lat1, lon1, lat2, lon2)

print("Matriz de distancias (km):")
print(matriz.round(0))

# Encontrar las dos más lejanas
max_dist = matriz.stack().idxmax()
print(f"\nCiudades más lejanas: {max_dist[0]} — {max_dist[1]}: {matriz.loc[max_dist]:.0f} km")

### Ejercicio 2 — Mapa de sucursales con > 3000 clientes

In [ ]:
df_alto = df[df["clientes"] > 3000]
print(f"Sucursales de alto volumen: {len(df_alto)}")
print(df_alto.groupby("ciudad").size().sort_values(ascending=False))

### Ejercicio 3 — K-Means con K=3, 5 y 7

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for idx, k in enumerate([3, 5, 7]):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(coords)
    axes[idx].scatter(df["lon"], df["lat"], c=labels, cmap="tab10", s=50, alpha=0.8)
    axes[idx].set_title(f"K={k} zonas")
    axes[idx].set_xlabel("Longitud")
    axes[idx].set_ylabel("Latitud")
plt.suptitle("Clustering espacial con distintos K")
plt.tight_layout()
plt.show()